# 23 · PyTorch en pratique 1 : du dataset au modèle sauvegardé — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch23_pytorch_1/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch23_pytorch_1/03_notebook.ipynb`, créée par `python tools/start_chapter.py 23`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 23.Q1 à 23.Q12, 23.R1 à 23.R3, 23.1 à 23.8 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 23.12 à 23.16 | l'environnement et les tenseurs, Fashion-MNIST regardé, oublier la conversion ou la normalisation, tout le prétraitement au même endroit, le MLP du livre (*prochaine session de génération*) | | | |
| B | 23.17 à 23.20 | trois bugs d'inférence, ta fonction fit() et son historique, lire l'historique, l'analyse des erreurs (*prochaine session de génération*) | | | |
| C | 23.21 à 23.24 | des images « du monde réel », prédire, reprendre un entraînement interrompu, un modèle pré-entraîné de torchvision (*prochaine session de génération*) | | | |
| D | 23.25 à 23.29 | les callbacks en PyTorch, les plannings de learning rate, la patience de l'early stopping, la fiche modèle, le défi reproductibilité (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="23")  # your own library

## Objectifs et rappel express

- Préparer un dataset d'images pour PyTorch : formes `(N, C, H, W)`, type `float32`, normalisation avec les statistiques du train, `Dataset` et `DataLoader`.
- Écrire ta propre fonction `fit()` (boucle, validation, historique), puis analyser les erreurs du modèle et prédire sur de nouvelles images.
- Sauvegarder et recharger un modèle, reprendre un entraînement, réutiliser un modèle pré-entraîné, et écrire en PyTorch les callbacks de Keras (meilleur checkpoint, planning de learning rate, early stopping).

**Rappel express.** Indice plat d'un élément $(i, j, k)$ de forme $(D_1, D_2, D_3)$ : $i\,D_2 D_3 + j\,D_3 + k$. Normalisation : $x' = (x/255 - \mu_{\text{train}})/\sigma_{\text{train}}$. Couche dense : $(n_{in} + 1)\,n_{out}$ paramètres. Cross-entropy sur des logits : $L = -z_y + \ln \sum_j e^{z_j}$. Mini-batches par epoch : $\lceil N/B \rceil$. `LambdaLR` : $\eta_e = \eta_0 \cdot \lambda(e)$. Coût d'une epoch : $\approx 6PN$ opérations.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 23.1 à 23.7 et 🧮 23.8)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs lettres ou un ordre : entre guillemets (`"EF"`) ; une forme ou un triplet d'indices : entre guillemets (`"(5, 7, 1)"`) ; une liste : entre crochets (`[9, 9, 9]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la réflexion (🗣️ 23.9, ⚖️ 23.10, 📄 23.11) et l'entretien se corrigent avec `05_solutions.md`.

**23.Q1 — Pourquoi utiliser une bibliothèque plutôt que tout coder soi-même ?**

In [ ]:
# 23.Q1: the values you found on paper
answer_23_Q1a = ...  # a) True or False
answer_23_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_23_Q1c = ...  # c) the letter of your choice, e.g. "E"
answer_23_Q1d = ...  # d) True or False
answer_23_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_Q1a, answer_23_Q1b, answer_23_Q1c, answer_23_Q1d, answer_23_Q1e]):
    wb.check(f"23.Q1{letter}", answer)

**23.Q2 — Avertissement ou erreur ? Versions et dépréciations**

In [ ]:
# 23.Q2: the values you found on paper
answer_23_Q2a = ...  # a) the letter of your choice, e.g. "E"
answer_23_Q2b = ...  # b) True or False
answer_23_Q2c = ...  # c) True or False
answer_23_Q2d = ...  # d) the letter of your choice, e.g. "E"
answer_23_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_Q2a, answer_23_Q2b, answer_23_Q2c, answer_23_Q2d, answer_23_Q2e]):
    wb.check(f"23.Q2{letter}", answer)

**23.Q3 — Déboguer méthodiquement : réduire, isoler, reconstruire**

In [ ]:
# 23.Q3: the values you found on paper
answer_23_Q3a = ...  # a) the letter of your choice, e.g. "E"
answer_23_Q3b = ...  # b) True or False
answer_23_Q3c = ...  # c) a whole number
answer_23_Q3d = ...  # d) a whole number
answer_23_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_Q3a, answer_23_Q3b, answer_23_Q3c, answer_23_Q3d, answer_23_Q3e]):
    wb.check(f"23.Q3{letter}", answer)

**23.Q4 — Les trois sens du mot « modèle »**

In [ ]:
# 23.Q4: the values you found on paper
answer_23_Q4a = ...  # a) the letter of your choice, e.g. "E"
answer_23_Q4b = ...  # b) the letter of your choice, e.g. "E"
answer_23_Q4c = ...  # c) the letter of your choice, e.g. "E"
answer_23_Q4d = ...  # d) True or False
answer_23_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_Q4a, answer_23_Q4b, answer_23_Q4c, answer_23_Q4d, answer_23_Q4e]):
    wb.check(f"23.Q4{letter}", answer)

**23.Q5 — Tenseurs, arrays et type float32**

In [ ]:
# 23.Q5: the values you found on paper
answer_23_Q5a = ...  # a) a whole number
answer_23_Q5b = ...  # b) the letter of your choice, e.g. "E"
answer_23_Q5c = ...  # c) the letter of your choice, e.g. "E"
answer_23_Q5d = ...  # d) True or False
answer_23_Q5e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_23_Q5a, answer_23_Q5b, answer_23_Q5c, answer_23_Q5d, answer_23_Q5e]):
    wb.check(f"23.Q5{letter}", answer)

**23.Q6 — channels_first, channels_last et NCHW**

In [ ]:
# 23.Q6: the values you found on paper
answer_23_Q6a = ...  # a) a shape as a string, e.g. "(3, 5, 1)"
answer_23_Q6b = ...  # b) a shape as a string, e.g. "(3, 5, 1)"
answer_23_Q6c = ...  # c) the letter of your choice, e.g. "E"
answer_23_Q6d = ...  # d) a shape as a string, e.g. "(3, 5, 1)"
answer_23_Q6e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_Q6a, answer_23_Q6b, answer_23_Q6c, answer_23_Q6d, answer_23_Q6e]):
    wb.check(f"23.Q6{letter}", answer)

**23.Q7 — CPU, GPU, MPS, TPU : où calcule PyTorch ?**

In [ ]:
# 23.Q7: the values you found on paper
answer_23_Q7a = ...  # a) the letter of your choice, e.g. "E"
answer_23_Q7b = ...  # b) True or False
answer_23_Q7c = ...  # c) the letter of your choice, e.g. "E"
answer_23_Q7d = ...  # d) True or False
answer_23_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_Q7a, answer_23_Q7b, answer_23_Q7c, answer_23_Q7d, answer_23_Q7e]):
    wb.check(f"23.Q7{letter}", answer)

**23.Q8 — Iris, MNIST, Fashion-MNIST : les « hello world » du ML**

In [ ]:
# 23.Q8: the values you found on paper
answer_23_Q8a = ...  # a) a whole number
answer_23_Q8b = ...  # b) a whole number
answer_23_Q8c = ...  # c) the letter of your choice, e.g. "E"
answer_23_Q8d = ...  # d) True or False
answer_23_Q8e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_23_Q8a, answer_23_Q8b, answer_23_Q8c, answer_23_Q8d, answer_23_Q8e]):
    wb.check(f"23.Q8{letter}", answer)

**23.Q9 — Labels entiers, one-hot et logits**

In [ ]:
# 23.Q9: the values you found on paper
answer_23_Q9a = ...  # a) the letter of your choice, e.g. "E"
answer_23_Q9b = ...  # b) the letter of your choice, e.g. "E"
answer_23_Q9c = ...  # c) True or False
answer_23_Q9d = ...  # d) a shape as a string, e.g. "(3, 5, 1)"
answer_23_Q9e = ...  # e) a shape as a string, e.g. "(3, 5, 1)"

for letter, answer in zip("abcde", [answer_23_Q9a, answer_23_Q9b, answer_23_Q9c, answer_23_Q9d, answer_23_Q9e]):
    wb.check(f"23.Q9{letter}", answer)

**23.Q10 — fit() décortiqué : batch, epoch, validation, train() et eval()**

In [ ]:
# 23.Q10: the values you found on paper
answer_23_Q10a = ...  # a) True or False
answer_23_Q10b = ...  # b) the letter of your choice, e.g. "E"
answer_23_Q10c = ...  # c) the letter of your choice, e.g. "E"
answer_23_Q10d = ...  # d) True or False
answer_23_Q10e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_23_Q10a, answer_23_Q10b, answer_23_Q10c, answer_23_Q10d, answer_23_Q10e]):
    wb.check(f"23.Q10{letter}", answer)

**23.Q11 — Sauvegarder tout, les poids ou l'architecture ?**

In [ ]:
# 23.Q11: the values you found on paper
answer_23_Q11a = ...  # a) the letter of your choice, e.g. "E"
answer_23_Q11b = ...  # b) the letter of your choice, e.g. "E"
answer_23_Q11c = ...  # c) True or False
answer_23_Q11d = ...  # d) a whole number
answer_23_Q11e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_Q11a, answer_23_Q11b, answer_23_Q11c, answer_23_Q11d, answer_23_Q11e]):
    wb.check(f"23.Q11{letter}", answer)

**23.Q12 — 🕰️ Que devient le code Keras du livre en 2026 ?**

In [ ]:
# 23.Q12: the values you found on paper
answer_23_Q12a = ...  # a) the letter of your choice, e.g. "E"
answer_23_Q12b = ...  # b) the letter of your choice, e.g. "E"
answer_23_Q12c = ...  # c) True or False
answer_23_Q12d = ...  # d) True or False
answer_23_Q12e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_23_Q12a, answer_23_Q12b, answer_23_Q12c, answer_23_Q12d, answer_23_Q12e]):
    wb.check(f"23.Q12{letter}", answer)

**23.R1 — Ch. 22 : (N, T, D) et batch_first**

In [ ]:
# 23.R1: the values you found on paper
answer_23_R1a = ...  # a) a shape as a string, e.g. "(3, 5, 1)"
answer_23_R1b = ...  # b) a shape as a string, e.g. "(3, 5, 1)"
answer_23_R1c = ...  # c) a shape as a string, e.g. "(3, 5, 1)"
answer_23_R1d = ...  # d) True or False
answer_23_R1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_R1a, answer_23_R1b, answer_23_R1c, answer_23_R1d, answer_23_R1e]):
    wb.check(f"23.R1{letter}", answer)

**23.R2 — Ch. 20 : Les cinq lignes d'une étape d'entraînement**

In [ ]:
# 23.R2: the values you found on paper
answer_23_R2a = ...  # a) the letters, e.g. "EF"
answer_23_R2b = ...  # b) True or False
answer_23_R2c = ...  # c) the letter of your choice, e.g. "E"
answer_23_R2d = ...  # d) True or False
answer_23_R2e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_23_R2a, answer_23_R2b, answer_23_R2c, answer_23_R2d, answer_23_R2e]):
    wb.check(f"23.R2{letter}", answer)

**23.R3 — Ch. 2 : Moyenne et écart-type du train pour normaliser les pixels**

In [ ]:
# 23.R3: the values you found on paper
answer_23_R3a = ...  # a) a number, 3 decimals
answer_23_R3b = ...  # b) a number, 3 decimals
answer_23_R3c = ...  # c) a number, 3 decimals
answer_23_R3d = ...  # d) True or False
answer_23_R3e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_23_R3a, answer_23_R3b, answer_23_R3c, answer_23_R3d, answer_23_R3e]):
    wb.check(f"23.R3{letter}", answer)

**Ex 23.1 — Remodeler 12 éléments : indices et ordre de lecture**

In [ ]:
# Ex 23.1: the values you found on paper
answer_23_1a = ...  # a) a letter as a string, e.g. "M"
answer_23_1b = ...  # b) a letter as a string, e.g. "M"
answer_23_1c = ...  # c) a letter as a string, e.g. "M"
answer_23_1d = ...  # d) the indices as a string, e.g. "(5, 7, 1)"
answer_23_1e = ...  # e) the letters, e.g. "EF"
answer_23_1f = ...  # f) a list of 4 whole numbers, e.g. [9, 9, 9, 9]
answer_23_1g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_23_1a, answer_23_1b, answer_23_1c, answer_23_1d, answer_23_1e, answer_23_1f, answer_23_1g]):
    wb.check(f"23.1{letter}", answer)

**Ex 23.2 — De NHWC à NCHW : où atterrit chaque pixel ?**

In [ ]:
# Ex 23.2: the values you found on paper
answer_23_2a = ...  # a) a whole number
answer_23_2b = ...  # b) a shape as a string, e.g. "(3, 5, 1)"
answer_23_2c = ...  # c) the indices as a string, e.g. "(5, 7, 1)"
answer_23_2d = ...  # d) a whole number
answer_23_2e = ...  # e) True or False
answer_23_2f = ...  # f) the letter of your choice, e.g. "E"
answer_23_2g = ...  # g) the letter of your choice, e.g. "E"
answer_23_2h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_23_2a, answer_23_2b, answer_23_2c, answer_23_2d, answer_23_2e, answer_23_2f, answer_23_2g, answer_23_2h]):
    wb.check(f"23.2{letter}", answer)

**Ex 23.3 — 623 290 paramètres : refaire le compte du livre**

In [ ]:
# Ex 23.3: the values you found on paper
answer_23_3a = ...  # a) a whole number
answer_23_3b = ...  # b) a whole number
answer_23_3c = ...  # c) a whole number
answer_23_3d = ...  # d) a number, 3 decimals
answer_23_3e = ...  # e) a whole number
answer_23_3f = ...  # f) a whole number
answer_23_3g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_23_3a, answer_23_3b, answer_23_3c, answer_23_3d, answer_23_3e, answer_23_3f, answer_23_3g]):
    wb.check(f"23.3{letter}", answer)

**Ex 23.4 — Une cross-entropy à la main : indice de classe ou one-hot**

In [ ]:
# Ex 23.4: the values you found on paper
answer_23_4a = ...  # a) a number, 3 decimals
answer_23_4b = ...  # b) a number, 3 decimals
answer_23_4c = ...  # c) a number, 3 decimals
answer_23_4d = ...  # d) True or False
answer_23_4e = ...  # e) a number, 3 decimals
answer_23_4f = ...  # f) a number, 3 decimals
answer_23_4g = ...  # g) a number, 3 decimals

for letter, answer in zip("abcdefg", [answer_23_4a, answer_23_4b, answer_23_4c, answer_23_4d, answer_23_4e, answer_23_4f, answer_23_4g]):
    wb.check(f"23.4{letter}", answer)

**Ex 23.5 — Batches par epoch et nombre de mises à jour**

In [ ]:
# Ex 23.5: the values you found on paper
answer_23_5a = ...  # a) a whole number
answer_23_5b = ...  # b) a whole number
answer_23_5c = ...  # c) a whole number
answer_23_5d = ...  # d) a whole number
answer_23_5e = ...  # e) a whole number
answer_23_5f = ...  # f) a whole number
answer_23_5g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_23_5a, answer_23_5b, answer_23_5c, answer_23_5d, answer_23_5e, answer_23_5f, answer_23_5g]):
    wb.check(f"23.5{letter}", answer)

**Ex 23.6 — Early stopping à la main : patience et min_delta**

In [ ]:
# Ex 23.6: the values you found on paper
answer_23_6a = ...  # a) a whole number
answer_23_6b = ...  # b) a whole number
answer_23_6c = ...  # c) a whole number
answer_23_6d = ...  # d) a whole number
answer_23_6e = ...  # e) a whole number
answer_23_6f = ...  # f) a whole number
answer_23_6g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_23_6a, answer_23_6b, answer_23_6c, answer_23_6d, answer_23_6e, answer_23_6f, answer_23_6g]):
    wb.check(f"23.6{letter}", answer)

**Ex 23.7 — Plannings de learning rate : le planning du livre avec LambdaLR**

In [ ]:
# Ex 23.7: the values you found on paper
answer_23_7a = ...  # a) a number, 2 decimals
answer_23_7b = ...  # b) a whole number
answer_23_7c = ...  # c) a number, 4 decimals
answer_23_7d = ...  # d) a whole number
answer_23_7e = ...  # e) a number, 4 decimals
answer_23_7f = ...  # f) a number, 4 decimals
answer_23_7g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_23_7a, answer_23_7b, answer_23_7c, answer_23_7d, answer_23_7e, answer_23_7f, answer_23_7g]):
    wb.check(f"23.7{letter}", answer)

**Ex 23.8 — Fermi : durée d'une epoch et poids d'un checkpoint**

In [ ]:
# Ex 23.8: the values you found on paper
answer_23_8a = ...  # a) a number, 1 decimal
answer_23_8b = ...  # b) a number, 1 decimal
answer_23_8c = ...  # c) a number, 1 decimal
answer_23_8d = ...  # d) a number, 1 decimal
answer_23_8e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_23_8a, answer_23_8b, answer_23_8c, answer_23_8d, answer_23_8e]):
    wb.check(f"23.8{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu préparer des images pour PyTorch (forme, type, normalisation, `DataLoader`) et expliquer pourquoi `reshape` ne remplace pas `permute` ?
2. Sais-tu écrire une boucle d'entraînement avec validation, et dire ce que font `model.eval()` et `torch.inference_mode()` ?
3. Sais-tu sauvegarder un modèle pour le reprendre ou le livrer, avec son prétraitement, et programmer un meilleur checkpoint, un planning et un early stopping ?

**Pour aller plus loin** : les tutoriels « Learn the Basics » de PyTorch, cités dans la fiche. La suite : le ch. 24, PyTorch en pratique 2 (CNN, RNN, augmentation de données).

*Les exercices de code 23.12 à 23.29 (parties A à D : le pipeline Fashion-MNIST, ta fonction `fit()`, la sauvegarde, le modèle pré-entraîné et les callbacks) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 23` pour obtenir le notebook complet.*